Tugas 3 Nama:Ziffy Hilya NPM: 2505060008

In [8]:
# Sel ini menghasilkan TIGA dataset cabang (kota) terpisah untuk Tugas Mandiri Pertemuan 3
import numpy as np
import pandas as pd

kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-08-01", "2026-08-31", freq="D")

cabang_kota = {"Magelang": 101, "Yogyakarta": 202, "Semarang": 303}

for kota, seed in cabang_kota.items():
    np.random.seed(seed)  # seed berbeda tiap kota agar datanya bervariasi, namun tetap konsisten/reproducible
    n = 200
    data_cabang = {
        "order_id": [f"{kota[:3].upper()}-{2000 + i}" for i in range(n)],
        "tanggal": np.random.choice(tanggal_range, size=n),
        "kategori": np.random.choice(kategori_list, size=n, p=[0.25, 0.25, 0.20, 0.15, 0.15]),
        "unit_terjual": np.random.randint(1, 8, size=n),
        "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000, 250000], size=n),
        "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    }
    df_cabang = pd.DataFrame(data_cabang)
    df_cabang["kota"] = kota
    nama_file = f"transaksi_{kota.lower()}.csv"
    df_cabang.to_csv(nama_file, index=False)
    print(f"Berkas '{nama_file}' berhasil dibuat: {df_cabang.shape[0]} baris")

print("\nKetiga berkas CSV cabang siap digunakan untuk Tugas Mandiri.")

Berkas 'transaksi_magelang.csv' berhasil dibuat: 200 baris
Berkas 'transaksi_yogyakarta.csv' berhasil dibuat: 200 baris
Berkas 'transaksi_semarang.csv' berhasil dibuat: 200 baris

Ketiga berkas CSV cabang siap digunakan untuk Tugas Mandiri.


In [9]:
!hdfs dfs -mkdir -p /user/Ziffy2/ecommerce/raw
!hdfs dfs -mkdir -p /user/Ziffy2/ecommerce/processed
!hdfs dfs -ls /user/Ziffy2/ecommerce

Found 2 items
drwxr-xr-x   - Ziffy2 supergroup          0 2026-09-10 01:31 /user/Ziffy2/ecommerce/processed
drwxr-xr-x   - Ziffy2 supergroup          0 2026-09-10 01:37 /user/Ziffy2/ecommerce/raw


In [11]:
!hdfs dfs -put transaksi_magelang.csv transaksi_yogyakarta.csv transaksi_semarang.csv /user/Ziffy2/ecommerce/raw
!hdfs dfs -ls /user/Ziffy2/ecommerce/raw

put: `/user/Ziffy2/ecommerce/raw/transaksi_semarang.csv': File exists
Found 3 items
-rw-r--r--   1 Ziffy2 supergroup      12329 2026-09-10 01:51 /user/Ziffy2/ecommerce/raw/transaksi_magelang.csv
-rw-r--r--   1 Ziffy2 supergroup      12154 2026-09-10 01:37 /user/Ziffy2/ecommerce/raw/transaksi_semarang.csv
-rw-r--r--   1 Ziffy2 supergroup      12684 2026-09-10 01:51 /user/Ziffy2/ecommerce/raw/transaksi_yogyakarta.csv


In [12]:
# Mengunduh ketiga berkas 
!hdfs dfs -get /user/Ziffy2/ecommerce/raw/transaksi_magelang.csv magelang_dari_hdfs.csv
!hdfs dfs -get /user/Ziffy2/ecommerce/raw/transaksi_semarang.csv semarang_dari_hdfs.csv
!hdfs dfs -get /user/Ziffy2/ecommerce/raw/transaksi_yogyakarta.csv yogyakarta_dari_hdfs.csv

In [13]:
import pandas as pd
df_magelang = pd.read_csv("magelang_dari_hdfs.csv")
df_semarang = pd.read_csv("semarang_dari_hdfs.csv")
df_yogyakarta = pd.read_csv("yogyakarta_dari_hdfs.csv")

#gabungkan ketiga data frame
df_gabungan = pd.concat ([df_magelang, df_semarang, df_yogyakarta], ignore_index=True)
df_gabungan

,order_id,tanggal,kategori,unit_terjual,harga_satuan,metode_pembayaran,kota
0,MAG-2000,2026-08-12,Fashion,1,50000,Transfer Bank,Magelang
1,MAG-2001,2026-08-18,Elektronik,7,25000,COD,Magelang
2,MAG-2002,2026-08-07,Elektronik,7,25000,E-Wallet,Magelang
3,MAG-2003,2026-08-24,Rumah Tangga,6,25000,COD,Magelang
4,MAG-2004,2026-08-30,Fashion,7,100000,Transfer Bank,Magelang
...,...,...,...,...,...,...,...
595,YOG-2195,2026-08-25,Rumah Tangga,4,250000,Kartu Kredit,Yogyakarta
596,YOG-2196,2026-08-18,Fashion,1,150000,COD,Yogyakarta
597,YOG-2197,2026-08-27,Makanan & Minuman,4,25000,Kartu Kredit,Yogyakarta
598,YOG-2198,2026-08-07,Rumah Tangga,1,25000,COD,Yogyakarta


In [14]:
df_gabungan["kota"].value_counts()

kota
Magelang      200
Semarang      200
Yogyakarta    200
Name: count, dtype: int64

In [15]:
#menambah kolom total_pendapatan
df_gabungan["total_pendapatan"] = df_gabungan["unit_terjual"] * df_gabungan["harga_satuan"]
df_gabungan.head()

,order_id,tanggal,kategori,unit_terjual,harga_satuan,metode_pembayaran,kota,total_pendapatan
0,MAG-2000,2026-08-12,Fashion,1,50000,Transfer Bank,Magelang,50000
1,MAG-2001,2026-08-18,Elektronik,7,25000,COD,Magelang,175000
2,MAG-2002,2026-08-07,Elektronik,7,25000,E-Wallet,Magelang,175000
3,MAG-2003,2026-08-24,Rumah Tangga,6,25000,COD,Magelang,150000
4,MAG-2004,2026-08-30,Fashion,7,100000,Transfer Bank,Magelang,700000


In [16]:
#tabel ringkasan
ringkasan_kota_kategori = df_gabungan.groupby(["kota", "kategori"])["total_pendapatan"].sum().reset_index()
ringkasan_kota_kategori

,kota,kategori,total_pendapatan
0,Magelang,Elektronik,18775000
1,Magelang,Fashion,27750000
2,Magelang,Kesehatan & Kecantikan,17375000
3,Magelang,Makanan & Minuman,17525000
4,Magelang,Rumah Tangga,12200000
5,Semarang,Elektronik,21425000
6,Semarang,Fashion,26425000
7,Semarang,Kesehatan & Kecantikan,13525000
8,Semarang,Makanan & Minuman,19750000
9,Semarang,Rumah Tangga,10975000


In [18]:
#menyimpan dua berkas di disk lokal
df_gabungan.to_csv("data_gabungan_bersih.csv", index=False)
ringkasan_kota_kategori.to_csv("ringkasan_kota_kategori.csv", index=False)
print("Kedua berkas berhasil disimpan di disk lokal.")

Kedua berkas berhasil disimpan di disk lokal.


In [20]:
#mengunggah kedua berkas hasil olahan
!hdfs dfs -put data_gabungan_bersih.csv /user/Ziffy2/ecommerce/processed
!hdfs dfs -put ringkasan_kota_kategori.csv /user/Ziffy2/ecommerce/processed

#membuktikan kedua berkas berhasil terunggah
!hdfs dfs -ls /user/Ziffy2/ecommerce/processed

put: `/user/Ziffy2/ecommerce/processed/data_gabungan_bersih.csv': File exists
put: `/user/Ziffy2/ecommerce/processed/ringkasan_kota_kategori.csv': File exists
Found 2 items
-rw-r--r--   1 Ziffy2 supergroup      41257 2026-09-10 02:29 /user/Ziffy2/ecommerce/processed/data_gabungan_bersih.csv
-rw-r--r--   1 Ziffy2 supergroup        530 2026-09-10 02:29 /user/Ziffy2/ecommerce/processed/ringkasan_kota_kategori.csv


Apa keuntungan menyimpan data mentah (raw) terpisah dari data olahan (processed) di HDFS, dibandingkan menyimpan semuanya bercampur dalam
satu folder?

Memisahkan data raw dan processed di HDFS memiliki beberapa keuntungan, terutama dalam menjaga data agar lebih rapi dan mudah dikelola. 
Data raw merupakan data asli yang belum mengalami perubahan, sehingga tetap dapat digunakan kembali apabila diperlukan untuk proses 
pengolahan ulang. Sementara itu, data processed berisi data yang sudah dibersihkan, diubah, atau diproses sesuai kebutuhan. 
Dengan memisahkan keduanya, pengguna akan lebih mudah mencari dan membedakan data berdasarkan fungsinya. Selain itu, pemisahan folder 
juga dapat mengurangi risiko data asli tertimpa atau tercampur dengan data hasil pengolahan. Struktur penyimpanan seperti ini juga membuat 
proses pengelolaan, pemantauan, dan pengolahan data di HDFS menjadi lebih teratur dan efisien.

In [22]:
#mengecek ukuran total disk
!hdfs dfs -du -h /user/Ziffy2/ecommerce

#menghitung jumlah file & folder di dalamnya
!hdfs dfs -count /user/Ziffy2/ecommerce

40.8 K  40.8 K  /user/Ziffy2/ecommerce/processed
36.3 K  36.3 K  /user/Ziffy2/ecommerce/raw
           3            5              78954 /user/Ziffy2/ecommerce


In [25]:
#melihat 5 baris pertama,tanpa didownload
!hdfs dfs -cat /user/Ziffy2/ecommerce/processed/ringkasan_kota_kategori.csv | head -5

kota,kategori,total_pendapatan
Magelang,Elektronik,18775000
Magelang,Fashion,27750000
Magelang,Kesehatan & Kecantikan,17375000
Magelang,Makanan & Minuman,17525000


In [26]:
#melihat detail block dan replication factor dari salah satu file
!hdfs fsck /user/Ziffy2/ecommerce/raw/transaksi_magelang.csv -files -blocks

Connecting to namenode via http://localhost:9870/fsck?ugi=Ziffy2&files=1&blocks=1&path=%2Fuser%2FZiffy2%2Fecommerce%2Fraw%2Ftransaksi_magelang.csv
FSCK started by Ziffy2 (auth:SIMPLE) from /127.0.0.1 for path /user/Ziffy2/ecommerce/raw/transaksi_magelang.csv at Thu Sep 10 04:07:15 UTC 2026

/user/Ziffy2/ecommerce/raw/transaksi_magelang.csv 12329 bytes, replicated: replication=1, 1 block(s):  OK
0. BP-420316264-10.0.2.15-1788995956288:blk_1073741827_1003 len=12329 Live_repl=1


Status: HEALTHY
 Number of data-nodes:	1
 Number of racks:		1
 Total dirs:			0
 Total symlinks:		0

Replicated Blocks:
 Total size:	12329 B
 Total files:	1
 Total blocks (validated):	1 (avg. block size 12329 B)
 Minimally replicated blocks:	1 (100.0 %)
 Over-replicated blocks:	0 (0.0 %)
 Under-replicated blocks:	0 (0.0 %)
 Mis-replicated blocks:		0 (0.0 %)
 Default replication factor:	1
 Average block replication:	1.0
 Missing blocks:		0
 Corrupt blocks:		0
 Missing replicas:		0 (0.0 %)
 Blocks queued for replica

In [27]:
#melihat status keseluruhan cluster HDFS
!hdfs dfsadmin -report

Configured Capacity: 52518420480 (48.91 GB)
Present Capacity: 22909288448 (21.34 GB)
DFS Remaining: 22909136896 (21.34 GB)
DFS Used: 151552 (148 KB)
DFS Used%: 0.00%
Replicated Blocks:
	Under replicated blocks: 0
	Blocks with corrupt replicas: 0
	Missing blocks: 0
	Missing blocks (with replication factor 1): 0
	Low redundancy blocks with highest priority to recover: 0
	Pending deletion blocks: 0
Erasure Coded Block Groups: 
	Low redundancy block groups: 0
	Block groups with corrupt internal blocks: 0
	Missing block groups: 0
	Low redundancy blocks with highest priority to recover: 0
	Pending deletion blocks: 0

-------------------------------------------------
Live datanodes (1):

Name: 127.0.0.1:9866 (localhost.localdomain)
Hostname: 10.0.2.15
Decommission Status : Normal
Configured Capacity: 52518420480 (48.91 GB)
DFS Used: 151552 (148 KB)
Non DFS Used: 26908160000 (25.06 GB)
DFS Remaining: 22909136896 (21.34 GB)
DFS Used%: 0.00%
DFS Remaining%: 43.62%
Configured Cache Capacity: 0 (0

In [28]:
#melihat info detail: waktu modifikasi,ukuran, replication, block size
!hdfs dfs -stat "%n | ukuran: %b bytes |Replikasi: %r | Terakhir diubah: %y" /user/Ziffy2/ecommerce/raw/transaksi_magelang.csv

transaksi_magelang.csv | ukuran: 12329 bytes |Replikasi: 1 | Terakhir diubah: 2026-09-10 01:51:39
